# Data quality check
Here we check how good the Alzheimer's trials table actually is before doing any analysis: what's missing, and if the column types are right.

In [1]:
import pandas as pd
import sys
sys.path.append("..")
df = pd.read_csv("../data/processed/alzheimer.csv")

In [2]:
from src.fixed_data import fix_data
df = fix_data(df)


fix_data (src/fixed_data.py) turns enrollment into a whole number and the dates into real dates.

In [3]:
df.dtypes

nct_id                        str
title                         str
status                        str
phases                        str
study_type                    str
startdate          datetime64[us]
why_stopped                   str
enrollment                  Int64
completion_date    datetime64[us]
sponsor_type                  str
has_results                  bool
dtype: object

Now we check what % of each column is empty.

In [4]:
((df.isna().sum() / df.shape[0]) * 100).round(2)

nct_id              0.00
title               0.00
status              0.00
phases             57.90
study_type          0.00
startdate           0.78
why_stopped        91.34
enrollment          0.85
completion_date     2.07
sponsor_type        0.00
has_results         0.00
dtype: float64

why_stopped shows 91.32% empty, but that's misleading because most trials never stopped, so they have nothing to report.
So we filter only the TERMINATED trials (mask on status) and check how many of those have no reason.
Result: 14.8% of the 304 terminated trials have no why_stopped, so 85% of terminated trials report a reason.

In [5]:
mask = df["status"] == "TERMINATED"
terminated = df[mask]
((terminated["why_stopped"].isna().sum()/terminated.shape[0]) * 100).round(2)

np.float64(14.8)

We now focus on the 57.84% given by phases, as those are likely observational studies, so there are no phases so we verify by adding study_type

In [6]:
mask = df["study_type"] == "OBSERVATIONAL"
observational = df[mask]
((observational["phases"].isna().sum()/observational.shape[0]) * 100).round(2)

np.float64(100.0)

Some dates only have year and month (like 2015-09). A date needs a day, so we put the 15th, the middle of the month, so we're never more than ~15 days off. We didn't drop them because that would lose real trials and could bias the data.

# Conclusion
- **why_stopped:** the 91.32% empty is misleading. Among the 304 terminated trials only 14.8% have no reason, so 85% of terminated trials report why they stopped.
- **phases:** the 57.84% empty is expected. 100% of observational trials have no phase because they don't test a treatment.
- **dates:** year-month dates got the 15th so they're never more than ~15 days off, and none were dropped. Start and completion dates are now real dates and enrollment is a whole number.
- **small gaps:** startdate (0.78%), enrollment (0.86%) and completion_date (2.08%) have a few missing values, small enough to live with.